# DCGAN — convolutions make GAN training stable

> Tutorial pair for [`dcgan.py`](dcgan.py).

## 1. Intuition
Vanilla GANs on images with fully-connected nets are fragile. **DCGAN** keeps
the exact same adversarial *game* but swaps the architecture for convolutions
and a recipe of choices (strided convs instead of pooling, BatchNorm, ReLU in
G / LeakyReLU in D, Tanh output) that empirically tames the unstable two-player
optimization. Here we shrink it to tiny 1x16x16 images so it runs in seconds.

## 2. Concept (the slide)
- **Generator** upsamples a noise vector to an image with `ConvTranspose2d`:
  $1\times1 \to 4\times4 \to 8\times8 \to 16\times16$.
- **Discriminator** mirrors it with strided `Conv2d` down to a single logit.
- **DCGAN guidelines:** no pooling (use strided / fractional-strided convs),
  BatchNorm in both nets (except D's input and G's output), ReLU in G with a
  **Tanh** output, LeakyReLU in D, weights $\sim\mathcal N(0, 0.02)$.
- The **loss is unchanged** from vanilla GAN — DCGAN is an *architecture* result.

## 3. Math derivation — same game, better-conditioned gradients

DCGAN optimizes the **same** non-saturating GAN objective as the vanilla model.
The discriminator maximizes
$$\mathcal L_D=\mathbb E_{x\sim p_{\text{data}}}[\log D(x)]
 +\mathbb E_{z\sim p_z}[\log(1-D(G(z)))],$$
and the generator uses the non-saturating loss
$$\mathcal L_G=-\,\mathbb E_{z\sim p_z}\big[\log D(G(z))\big].$$
With the optimal discriminator $D^\star(x)=\frac{p_{\text{data}}(x)}{p_{\text{data}}(x)+p_g(x)}$
this still drives $p_g\to p_{\text{data}}$ via the Jensen–Shannon divergence.

**So what does DCGAN change?** Not the objective but the *parameterization* of
$G$ and $D$, which changes the conditioning of the gradients:

- A **transposed convolution** with stride $s$ is the adjoint of a strided
  convolution; it learns its upsampling kernel rather than fixing it, so $G$ can
  synthesize spatial structure directly.
- **BatchNorm** normalizes each pre-activation $\hat h=\frac{h-\mu_B}{\sqrt{\sigma_B^2+\epsilon}}$
  then rescales $\gamma\hat h+\beta$. This keeps activations in a well-scaled
  regime, preventing one network from collapsing the other early — the dominant
  GAN failure mode. It is omitted at D's input and G's Tanh output to avoid
  constraining the data statistics.
- **Tanh** at G's output matches data scaled to $[-1,1]$, giving bounded,
  symmetric gradients.

Net effect: the JS-divergence game is the same, but $\nabla_\theta\mathcal L$ is
far better conditioned, so training converges where the MLP GAN oscillates.

## 4. Generator / key component

In [ ]:
# ===== actual implementation from dcgan.py =====
from __future__ import annotations

import numpy as np

import torch

import torch.nn as nn

SEED = 0

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def make_images(n: int = 256, size: int = 16, seed: int = SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    cx = cy = (size - 1) / 2
    radial = -((xx - cx) ** 2 + (yy - cy) ** 2)
    radial = radial / radial.min()  # normalize roughly to [0, 1]
    imgs = np.empty((n, 1, size, size), dtype=np.float32)
    for i in range(n):
        img = 0.3 * radial.copy()
        # bright square in one of 4 corners (the "structure" G must learn)
        c = rng.integers(0, 4)
        h = size // 3
        ys = 0 if c < 2 else size - h
        xs = 0 if c % 2 == 0 else size - h
        img[ys:ys + h, xs:xs + h] += 0.7
        img += 0.03 * rng.standard_normal((size, size)).astype(np.float32)
        imgs[i, 0] = np.clip(img, 0, 1)
    return (imgs * 2 - 1).astype(np.float32)

class Generator(nn.Module):
    def __init__(self, noise_dim: int = 32, ngf: int = 32, img_ch: int = 1):
        super().__init__()
        self.noise_dim = noise_dim
        self.net = nn.Sequential(
            # 1x1 -> 4x4
            nn.ConvTranspose2d(noise_dim, ngf * 2, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            # 4x4 -> 8x8
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            # 8x8 -> 16x16
            nn.ConvTranspose2d(ngf, img_ch, 4, 2, 1, bias=False),
            nn.Tanh(),  # outputs in [-1, 1]
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z.view(z.size(0), self.noise_dim, 1, 1))

## 5. Trainer / losses

In [ ]:
# ===== actual implementation from dcgan.py =====
class Discriminator(nn.Module):
    def __init__(self, ndf: int = 32, img_ch: int = 1):
        super().__init__()
        self.net = nn.Sequential(
            # 16x16 -> 8x8 (no BatchNorm on the input layer)
            nn.Conv2d(img_ch, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, True),
            # 8x8 -> 4x4
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),
            # 4x4 -> 1x1 logit
            nn.Conv2d(ndf * 2, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).view(x.size(0), 1)

def _weights_init(m: nn.Module) -> None:
    """DCGAN init: weights ~ N(0, 0.02), BatchNorm gamma ~ N(1, 0.02)."""
    cls = m.__class__.__name__
    if "Conv" in cls:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in cls:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)

class DCGANTorch:
    def __init__(self, noise_dim: int = 32, img_size: int = 16, lr: float = 2e-4):
        torch.manual_seed(SEED)
        self.dev = get_device()
        self.noise_dim, self.img_size = noise_dim, img_size
        self.G = Generator(noise_dim).to(self.dev).apply(_weights_init)
        self.D = Discriminator().to(self.dev).apply(_weights_init)
        self.optG = torch.optim.Adam(self.G.parameters(), lr=lr, betas=(0.5, 0.999))
        self.optD = torch.optim.Adam(self.D.parameters(), lr=lr, betas=(0.5, 0.999))
        self.bce = nn.BCEWithLogitsLoss()

    def fit(self, real: np.ndarray, steps: int = 400, batch: int = 64):
        real = torch.as_tensor(real, dtype=torch.float32, device=self.dev)
        self.d_hist, self.g_hist = [], []
        for _ in range(steps):
            idx = torch.randint(0, len(real), (batch,), device=self.dev)
            x = real[idx]
            # --- D step: real -> 1, fake -> 0 ---
            z = torch.randn(batch, self.noise_dim, device=self.dev)
            fake = self.G(z).detach()
            lossD = self.bce(self.D(x), torch.ones(batch, 1, device=self.dev)) \
                + self.bce(self.D(fake), torch.zeros(batch, 1, device=self.dev))
            self.optD.zero_grad(); lossD.backward(); self.optD.step()
            # --- G step: non-saturating, label fakes as real ---
            z = torch.randn(batch, self.noise_dim, device=self.dev)
            lossG = self.bce(self.D(self.G(z)), torch.ones(batch, 1, device=self.dev))
            self.optG.zero_grad(); lossG.backward(); self.optG.step()
            self.d_hist.append(lossD.item()); self.g_hist.append(lossG.item())
        return self

    @torch.no_grad()
    def generate(self, n: int) -> np.ndarray:
        self.G.eval()
        z = torch.randn(n, self.noise_dim, device=self.dev)
        out = self.G(z).cpu().numpy()
        self.G.train()
        return out

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    real = make_images(256)
    print(f"data: {real.shape}, range [{real.min():.2f}, {real.max():.2f}]")

    gan = DCGANTorch().fit(real, steps=400, batch=64)
    fake = gan.generate(64)

    d0, d1 = np.mean(gan.d_hist[:50]), np.mean(gan.d_hist[-50:])
    print(f"D loss trend: {d0:.3f} -> {d1:.3f}")
    print(f"G loss trend: {np.mean(gan.g_hist[:50]):.3f} -> {np.mean(gan.g_hist[-50:]):.3f}")

    # quality proxy: per-pixel mean/std of fakes should approach the real data's
    print(f"real  pixel mean={real.mean():.3f} std={real.std():.3f}")
    print(f"fake  pixel mean={fake.mean():.3f} std={fake.std():.3f}")
    # the bright corner square should give fakes high-intensity corners
    corner = fake[:, 0, :5, :5].mean()
    print(f"fake top-left 5x5 mean={corner:.3f} (real corners are bright)")

## 6. Train

In [ ]:
demo()

## 7. Visualization

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import dcgan as M

real = M.make_images(256)
gan = M.DCGANTorch().fit(real, steps=200, batch=64)  # lighter retrain just for the picture
fake = gan.generate(8)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for j in range(8):
    axes[0, j].imshow(real[j, 0], cmap="gray", vmin=-1, vmax=1); axes[0, j].axis("off")
    axes[1, j].imshow(fake[j, 0], cmap="gray", vmin=-1, vmax=1); axes[1, j].axis("off")
axes[0, 0].set_ylabel("real", rotation=0, labelpad=20); axes[1, 0].set_ylabel("fake", rotation=0, labelpad=20)
fig.suptitle("DCGAN: real (top) vs generated (bottom)")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- DCGAN is an **architecture** contribution: the game (JS divergence) is identical
  to vanilla GAN, but strided convs + BatchNorm + Tanh make gradients well-behaved.
- Pitfalls: BatchNorm on D's *input* or G's *output* hurts; forgetting to scale
  data to $[-1,1]$ fights the Tanh; too-large LR still triggers mode collapse.
- Next steps: **WGAN/WGAN-GP** change the *objective* for smoother gradients;
  **conditional GAN** adds labels for controllable generation.